## Install Required Libraries

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
!pip install -q datasets transformers evaluate sentence-transformers openai pandas numpy tqdm python-dotenv  google-generativeai requests tqdm -q sacrebleu rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.5 MB/s eta 0:00:00


In [ ]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 11.4 MB/s eta 0:00:00


In [ ]:
!pip install bart-score

ERROR: Could not find a version that satisfies the requirement bart-score (from versions: none)
ERROR: No matching distribution found for bart-score


## Import Required Libraries

In [ ]:
import os
import csv
import json
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import evaluate
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# Other Packages
import pandas as pd
import numpy as np
import os
from huggingface_hub import login
import getpass
from google.colab import userdata
from tqdm.auto import tqdm
import os
import csv

## Display Settings

In [ ]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', None)           # Don't wrap to next line
pd.set_option('display.max_colwidth', None)    # Show full column content

## Hugginface Login

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in using environment variable")
else:
    print("No HF_TOKEN found")

Logged in using environment variable


## OpenAI Login

In [ ]:
from openai import OpenAI
import os

# OpenAI API key
api_key = userdata.get('OPENAI_API_KEY')

os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)

## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Load the test split and choose sample size

In [ ]:
dataset_name = "Lakshan2003/customer-support-context-summary-50k"
split_name   = "test"
num_samples  = None  # Set to an integer to evaluate a subset (in order) or None for all

test_dataset = load_dataset(dataset_name, split=split_name)
if num_samples is not None:
    test_dataset = test_dataset.select(range(num_samples))  # Keep original order

print(f"Loaded {len(test_dataset)} test samples")

README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/5.82M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/35000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 10000 test samples


### Select and Load a LoRA‑finetuned model

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else None

# Load base model + processor with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.1-8B-Instruct",  # Base Llama 3.2
    max_seq_length = 1024,
    dtype = torch.float16,
    device_map="auto",
    load_in_4bit=not use_bf16  # fallback to 4-bit when no BF16
)


#  LoRA adapter
adapter_path = "Lakshan2003/Llama-3.1-8B-Instruct-customerservice-context-summary"
model = PeftModel.from_pretrained(model, adapter_path)

# Merge the adapter
model = model.merge_and_unload()

model.eval()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_1860/1080645981.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    

### Prompt Templat for the Model

In [ ]:
ft_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{instruction}<|eot_id|>

<|start_header_id|>user<|end_header_id|>

Conversation History:
{history}

<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>"""


# End of sentence special token
EOS_TOKEN = tokenizer.eos_token

### Generation Settings

In [ ]:
# Sampling parameters (top_p, top_k, etc.)
generation_config = {
    "max_new_tokens": 256,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,    # nucleus sampling
    "top_k": 50,
}

### Build Chat template function

In [ ]:
def build_prompt(example):
  # Accept dict rows or raw strings
  if isinstance(example, str):
      example = {"history_summary": example}
  elif not isinstance(example, dict):
      example = {"history_summary": str(example)}

  prompt = ft_prompt.format(
      instruction=example.get("instruction", ""),
      history=example.get("history_summary", "")
  )

  return prompt + EOS_TOKEN

#### Output file

In [ ]:
lora_adapter = "Llama-3.1_8B-instruct-customerservice-context-summary"
results_dir  = f"/content/drive/MyDrive/fyp-2025/Datasets/ContextSummaryTestData/{lora_adapter}"
results_path = os.path.join(results_dir, f"results_lora_customerservice{lora_adapter}.csv")
os.makedirs(results_dir, exist_ok=True)

In [ ]:
model.eval()
torch.set_grad_enabled(False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

In [ ]:
SAVE_COLS = [
    "conversation_id",
    "instruction",
    "conversation_history",
    "summarization_prompt",
    "history_summary",
    "client_question",
    "agent_answer",
    "refined_agent_answer",
    "generated_summary",
]

In [ ]:
def as_str(x):
    try:
        return str(x)
    except Exception:
        return ""

In [ ]:
# resume Logic
processed_ids, header_written = set(), False
if os.path.exists(results_path):
    try:
        prev = pd.read_csv(results_path)
        if "conversation_id" in prev.columns:
            processed_ids = set(prev["conversation_id"].astype(str))
        header_written = True
        print(f"Resuming from {len(processed_ids)} saved rows.")
    except Exception as e:
        print(f"CSV unreadable, starting fresh: {e}")

Resuming from 4544 saved rows.


In [ ]:
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
to_run = []
for ex in test_dataset:  # expects dict-like rows
    cid = as_str(ex.get("conversation_id", ""))
    if cid and cid in processed_ids:
        continue
    to_run.append(ex)
print(f"Pending: {len(to_run)}")

Pending: 5456


In [ ]:
if getattr(tokenizer, "pad_token", None) is None:
    tokenizer.pad_token = tokenizer.eos_token

eos_ids = [tokenizer.eos_token_id]
end = "<|end|>"
if hasattr(tokenizer, "get_vocab") and end in tokenizer.get_vocab():
    end_id = tokenizer.convert_tokens_to_ids(end)
    if end_id != tokenizer.eos_token_id:
        eos_ids = [tokenizer.eos_token_id, end_id]

In [ ]:
gen_batch_size = 32
pbar = tqdm(total=len(to_run), desc="Generating", unit="rec")

model.eval()
torch.set_grad_enabled(False)

for start in range(0, len(to_run), gen_batch_size):
    batch = to_run[start : start + gen_batch_size]

    prompts, metas = [], []
    for ex in batch:
        conversation_id        = as_str(ex.get("conversation_id", ""))
        instruction            = ex.get("instruction", "")
        conversation_history   = ex.get("conversation_history", "")
        summarization_prompt   = ex.get("summarization_prompt", "")
        history_summary        = ex.get("history_summary", "")
        client_question        = ex.get("client_question", "")
        agent_answer           = ex.get("agent_answer", "")
        refined_agent_answer   = ex.get("refined_agent_answer", "")

        # Build summarization prompt
        input_prompt = ft_prompt.format(
            instruction=summarization_prompt,
            history=conversation_history
        )

        prompts.append(input_prompt)
        metas.append({
            "conversation_id": conversation_id,
            "instruction": instruction,
            "conversation_history": conversation_history,
            "summarization_prompt": summarization_prompt,
            "history_summary": history_summary,
            "client_question": client_question,
            "agent_answer": agent_answer,
            "refined_agent_answer": refined_agent_answer,
        })

    # Tokenization
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    ).to(model.device)

    # Generation
    with torch.no_grad():

        out = model.generate(
            **enc,
            **generation_config,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            use_cache=True,
        )

    # Extract generated portion only
    seqs = out.sequences
    start_idx = enc.input_ids.shape[1]

    gen_texts = [
        tokenizer.decode(seqs[i, start_idx:], skip_special_tokens=True).strip()
        for i in range(seqs.size(0))
    ]

    # Save batch results
    out_df = pd.DataFrame(
        [{**m, "generated_summary": g} for m, g in zip(metas, gen_texts)],
        columns=SAVE_COLS
    )

    out_df.to_csv(
        results_path,
        mode="a",
        header=not os.path.exists(results_path),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pbar.update(len(batch))

pbar.close()

Generating:   0%|          | 0/5456 [00:00<?, ?rec/s]

In [ ]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv(results_path)

# Convert to Dataset
dataset = Dataset.from_pandas(df)

# Define repo name with model + use case
repo_id = "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-evaldata"

# Push to Hub
dataset.push_to_hub(repo_id, private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 14.2MB / 14.2MB            

README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-evaldata/commit/4b6b8a40769e942032440f8ec0ace6c87bdc024d', commit_message='Upload dataset', commit_description='', oid='4b6b8a40769e942032440f8ec0ace6c87bdc024d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-evaldata', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-evaldata'), pr_revision=None, pr_num=None)

In [ ]:
df.head(1)

,conversation_id,instruction,conversation_history,summarization_prompt,history_summary,client_question,agent_answer,refined_agent_answer,generated_summary
0,2a347f66d0474bab9ba5664e96ad8386,"You are a professional call-center customer service agent working at **Unity Financial Group**.\nReview the conversation history and any provided context (if available).\nMake sure your response is consistent with the conversation history (names, issues, and actions already taken).\nIf no history is given, treat the client’s message as the start of the conversation.\nContinue the dialogue as the agent by giving a clear, helpful, and professional response.\nResponses should sound natural and human-like, like a real phone call, and usually be few short sentences.\nProvide more detail when the client’s request clearly requires it.","Agent: Hi there! Thank you for choosing Unity Financial Group. My name is Ressie, how can I assist you today?\nClient: Hi Ressie! I'm calling because I have some questions about my youth account. Can you help me with that?\nAgent: Ofsolutely, Melody. I'd be happy to help. Can you please verify your account number so I can pull up your information?\nClient: Sure, it's *insert account number].\nAgent: Great, thank you. So, what questions do you have about your youth account?","You are a professional conversation summarization assistant.\nGoal:\nProduce a clear, concise and factual summary of the conversation so far so that\nYOU, the same customer service agent handling this client, can accurately answer\ntheir next question.\nInclude only information explicitly stated:\n- Client’s issue or request and current status (explicitly mention client and\nagent names if present in the conversation)\n- Verification steps completed or pending\n- Exact names, account numbers or identifiers, dates, amounts and actions taken\nor agreed\n- Commitments, deadlines and follow-up actions\n- Current state of the conversation\nExclude:\nGreetings, filler dialogue, speculation, assumptions, or invented details.\nStyle:\nNeutral and professional. Vary sentence structure and phrasing to avoid repetition.\nOutput:\nOne coherent, detailed paragraph summarizing the conversation context.\n","Melody contacted Unity Financial Group's agent Ressie to inquire about her youth account. She provided her account number for verification, which Ressie confirmed before proceeding. The client’s specific questions about the account have not yet been addressed in the conversation so far.","Well, I was wondering if there are any fees associated with the account, And also, how do I go about accessing my funds?","Great questions, Melody. Let me check on that for you. [pause Okay, I've pulled up your account information. There are no monthly maintenance fees associated with our youth accounts, which is great news for And as for accessing your funds, you can use your debit card to make purchases or withdraw cash from any ATM. You can also transfer funds to other account through online banking or our mobile app.","Absolutely, Melody. There are no monthly maintenance fees for your youth account, which is great! To access your funds, you can use your debit card for purchases or ATM withdrawals, and you can also transfer money through online banking or our mobile app. If you have any other questions, feel free to ask!","The client, Melody, contacted Unity Financial Group seeking information about her youth account. Agent Ressie requested verification of Melody's account number, which she provided, and then offered assistance with her inquiries. No specific questions or issues regarding the account have been disclosed or addressed yet. The conversation is currently at the stage of gathering information from Melody."


# Lexical Overlap Scores Calculation

In [ ]:
import pandas as pd
from datasets import Dataset


df = pd.read_csv(results_path)
preds = df["generated_summary"].tolist()
refs  = df["history_summary"].tolist()

## Bleu Score Calculation

In [ ]:
# Lexical overlap metrics
bleu   = evaluate.load("bleu").compute(predictions=preds, references=[[r] for r in refs])["bleu"]

bleu

0.37909819505880415

## Google Bleu Score Calculation

In [ ]:
gbleu  = evaluate.load("google_bleu").compute(predictions=preds, references=refs)["google_bleu"]

gbleu

0.3929049765113995

## Meteor Score Calculation

In [ ]:
meteor = evaluate.load("meteor").compute(predictions=preds, references=refs)["meteor"]

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
print(meteor)

0.617548876955343


## Rougue Score Calculation

In [ ]:
rouge  = evaluate.load("rouge").compute(predictions=preds, references=refs)

rouge

{'rouge1': np.float64(0.6750811219858504),
 'rouge2': np.float64(0.43506603718111625),
 'rougeL': np.float64(0.548562554808514),
 'rougeLsum': np.float64(0.5485466372561574)}

### Summarized Lexical Overlap Results

In [ ]:
import pandas as pd

# Build a summary table
metrics_table = pd.DataFrame([
    {"metric": "BLEU",         "score": bleu},
    {"metric": "Google BLEU",  "score": gbleu},
    {"metric": "METEOR",       "score": meteor},
    {"metric": "ROUGE-1",      "score": rouge["rouge1"]},
    {"metric": "ROUGE-2",      "score": rouge["rouge2"]},
    {"metric": "ROUGE-L",      "score": rouge["rougeL"]},
])

In [ ]:
metrics_table

,metric,score
0,BLEU,0.379098
1,Google BLEU,0.392905
2,METEOR,0.617549
3,ROUGE-1,0.675081
4,ROUGE-2,0.435066
5,ROUGE-L,0.548563


In [ ]:
from datasets import Dataset

repo = "Lakshan2003/Lexical-Overlap-Results-Llama-3.1-8b-customerservice-context-summary"

hf_dataset = Dataset.from_pandas(metrics_table.reset_index(drop=True))

# Push to the hub
hf_dataset.push_to_hub(repo)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 1.35kB / 1.35kB            

CommitInfo(commit_url='https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-Llama-3.1-8b-customerservice-context-summary/commit/9f584b3b334493dfbdfaaf0f34d21a0982bf80cc', commit_message='Upload dataset', commit_description='', oid='9f584b3b334493dfbdfaaf0f34d21a0982bf80cc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Lakshan2003/Lexical-Overlap-Results-Llama-3.1-8b-customerservice-context-summary', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Lakshan2003/Lexical-Overlap-Results-Llama-3.1-8b-customerservice-context-summary'), pr_revision=None, pr_num=None)

## Semantic Similarity Metrics

In [ ]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=results_path, split="train")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'conversation_history', 'summarization_prompt', 'history_summary', 'client_question', 'agent_answer', 'refined_agent_answer', 'generated_summary'],
    num_rows: 10000
})

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Generate embeddings for Groundtruth and generated answer

In [ ]:
def embed_batch(batch):
    gen = [str(x) if x is not None else "" for x in batch["generated_summary"]]
    gt  = [str(x) if x is not None else "" for x in batch["history_summary"]]

    # normalize_embeddings=True => unit vectors, so cosine = dot product
    gen_emb = model.encode(gen, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    gt_emb  = model.encode(gt,  convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

    # store as lists
    return {
        "generated_summary_embedding_transformers": [e.tolist() for e in gen_emb],
        "ground_summary_embedding_transformers": [e.tolist() for e in gt_emb],
    }

ds = ds.map(embed_batch, batched=True, batch_size=256)
ds

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['conversation_id', 'instruction', 'conversation_history', 'summarization_prompt', 'history_summary', 'client_question', 'agent_answer', 'refined_agent_answer', 'generated_summary', 'generated_answer_embedding_transformers', 'ground_truth_embedding_transformers', 'generated_summary_embedding_transformers', 'ground_summary_embedding_transformers'],
    num_rows: 10000
})

In [ ]:
import numpy as np

def cosine_batch(batch):
    gen = np.array(batch["generated_summary_embedding_transformers"], dtype=np.float32)
    gt  = np.array(batch["ground_summary_embedding_transformers"], dtype=np.float32)

    # cosine = sum(gen * gt) (using normalized embeddings)
    cos = (gen * gt).sum(axis=1)
    return {"cosine_similarity": cos.tolist()}

ds = ds.map(cosine_batch, batched=True, batch_size=1024)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

### BERT Score

In [ ]:
import evaluate
bertscore = evaluate.load("bertscore")

def add_bertscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_summary"]]
    refs  = [str(x) if x else "" for x in batch["history_summary"]]
    result = bertscore.compute(predictions=preds, references=refs, lang="en")
    return {
        "bertscore_precision": result["precision"],
        "bertscore_recall": result["recall"],
        "bertscore_f1": result["f1"],
    }

ds = ds.map(add_bertscore, batched=True, batch_size=64)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### BART Score

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(device)
bart_model.eval()

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
def compute_bartscore(sources, targets):
    scores = []
    for src, tgt in zip(sources, targets):
        inputs = bart_tokenizer(src, return_tensors="pt", truncation=True, max_length=512).to(device)
        with bart_tokenizer.as_target_tokenizer():
            labels = bart_tokenizer(tgt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            loss = bart_model(**inputs, labels=labels["input_ids"]).loss
        scores.append(-loss.item())  # negative loss = BARTScore
    return scores

def add_bartscore(batch):
    preds = [str(x) if x else "" for x in batch["generated_summary"]]
    refs  = [str(x) if x else "" for x in batch["history_summary"]]
    ref_hyp = compute_bartscore(refs, preds)
    hyp_ref = compute_bartscore(preds, refs)
    mean_score = [(a + b) / 2 for a, b in zip(ref_hyp, hyp_ref)]
    return {
        "bartscore_ref_hyp": ref_hyp,
        "bartscore_hyp_ref": hyp_ref,
        "bartscore_mean": mean_score,
    }

In [ ]:
ds = ds.map(add_bartscore, batched=True, batch_size=128)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


### Semantic Similarity based Scores

In [ ]:
df = ds.to_pandas()

In [ ]:
print("Summary:")
print(f"Mean cosine similarity : {df['cosine_similarity'].mean():.4f}")
print(f"Mean BERTScore F1      : {df['bertscore_f1'].mean():.4f}")
print(f"Mean BARTScore (mean)  : {df['bartscore_mean'].mean():.4f}")

Summary:
Mean cosine similarity : 0.9264
Mean BERTScore F1      : 0.9346
Mean BARTScore (mean)  : -1.8701


In [ ]:
import os
import pandas as pd

MODEL_NAME = "Llama-3.1_8B-instruct"
SAVE_DIR = "/content/drive/MyDrive/fyp-2025/Datasets/Contex_Summary_Semantic_Lexical_Similarity_Results"

metrics_rows = [
    {"model": MODEL_NAME, "metric": "BLEU",        "value": float(bleu)},
    {"model": MODEL_NAME, "metric": "Google BLEU", "value": float(gbleu)},
    {"model": MODEL_NAME, "metric": "METEOR",      "value": float(meteor)},
    {"model": MODEL_NAME, "metric": "ROUGE-1",     "value": float(rouge["rouge1"])},
    {"model": MODEL_NAME, "metric": "ROUGE-2",     "value": float(rouge["rouge2"])},
    {"model": MODEL_NAME, "metric": "ROUGE-L",     "value": float(rouge["rougeL"])},
    {"model": MODEL_NAME, "metric": "Cosine Similarity (mean)", "value": float(df["cosine_similarity"].mean())},
    {"model": MODEL_NAME, "metric": "BERTScore F1 (mean)",      "value": float(df["bertscore_f1"].mean())},
    {"model": MODEL_NAME, "metric": "BARTScore (mean)",         "value": float(df["bartscore_mean"].mean())},
]

metrics_table = pd.DataFrame(metrics_rows)

os.makedirs(SAVE_DIR, exist_ok=True)
csv_path = f"{SAVE_DIR}/metrics_{MODEL_NAME}.csv"
metrics_table.to_csv(csv_path, index=False)

print(csv_path)

/content/drive/MyDrive/fyp-2025/Datasets/Contex_Summary_Semantic_Lexical_Similarity_Results/metrics_Llama-3.1_8B-instruct.csv


### LLM as a judge Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import time
import json
import pandas as pd
from anthropic import Anthropic

#### Claude API setup

In [ ]:
from google.colab import userdata
claude_api = userdata.get('claude_api')

In [ ]:
anthropic = Anthropic(api_key=claude_api)

In [ ]:
# Configuration
original_csv = "/content/drive/MyDrive/context_summarization_eval_samples/LLaMA-31-8B_1000_samples.csv"

processed_folder = "/content/drive/MyDrive/fyp-2025/Datasets/ContextSummarizationLLMJudge/"
processed_csv = processed_folder + "LLaMA-31-8B_context_summarization_eval.csv"

batch_size = 50
batch_pause = 1.5  # seconds

#### EVALUATION PROMPT TEMPLATE (outside loop)

In [ ]:
EVAL_PROMPT = """
You are an expert evaluator specializing in dialogue summarization for customer-service interactions.
Evaluate the Generated Context Summary using the Original Multi-Turn Conversation as reference,
along with the Full Conversation History provided as additional background.
Your task is to assess how accurately, faithfully and clearly the Generated Context Summary
represents the key information from the conversation.

Note:
The Generated Context Summary is the model output being evaluated.
The Full Conversation History is provided only as supporting reference to help you better
understand the original interaction in detail.

Context Inputs:
Full Conversation History: {conversation_history}
Generated Context Summary: {generated_summary}

Evaluation Criteria and Scoring (1-5 each):

1. Information Accuracy

This measures how correctly the summary captures the important factual details from the conversation,
such as names, account numbers, dates, amounts, actions taken, verification steps, and follow-up commitments.

Rating Scale

1 = Important facts are missing or incorrect. The summary does not reflect the conversation properly.
2 = Some correct details are included, but several important facts are missing or wrong.
3 = Most important facts are correct, but a few details are missing. Overall meaning is still understandable.
4 = Almost all important facts are correct with only very small missing details.
5 = All key facts from the conversation are clearly and correctly included with no mistakes.


2. Structural Clarity

This measures how clearly the summary is organized. It checks whether the client issue,
conversation status, and key actions are presented in a clear and logical way.

Rating Scale

1 = The summary is confusing and poorly organized. The issue and status are hard to understand.
2 = Some structure exists, but the main issue or status is not clearly explained.
3 = The summary is generally understandable, but some parts are not clearly presented.
4 = The summary is well organized. The issue, status, and actions are easy to identify.
5 = The summary is very clear and logically structured. All parts are easy to read and understand.


3. Faithfulness

This measures whether the summary only includes information from the original conversation and does not add new or incorrect details.

Rating Scale

1 = The summary contains incorrect or invented information that was not in the conversation.
2 = Some parts include unsupported assumptions or slightly incorrect interpretations.
3 = Mostly faithful to the conversation, but a small amount of unsupported detail may appear.
4 = Very faithful to the conversation with only minimal inferred details that do not change the meaning.
5 = Completely faithful. The summary only contains information that appears in the conversation with no added or distorted content.

Return your answer as valid JSON only.
Do not include any explanation, commentary, additional text, or markdown formatting.
Output must contain JSON only and nothing else. All the below keys and their judgement score
should be included in your json response. Strictly follow only below json output. Always provide
the score for all tasks in the json.

Output Format (return only JSON):
{{
  "Information-Accuracy": [1-5],
  "Structural-Clarity": [1-5],
  "Faithfulness": [1-5]
}}
"""

In [ ]:
# Load Original or Resume From Processed Copy
os.makedirs(processed_folder, exist_ok=True)

if os.path.exists(processed_csv):
    print("Existing processed file found. Resuming from previous progress...")
    df = pd.read_csv(processed_csv)
else:
    print("No processed copy found. Creating one...")
    df = pd.read_csv(original_csv)

    # Add scoring columns if missing
    score_cols = [
        "Information-Accuracy",
        "Structural-Clarity",
        "Faithfulness",
    ]
    for c in score_cols:
        if c not in df.columns:
            df[c] = ""

    df.to_csv(processed_csv, index=False, encoding="utf-8-sig")

print("Loaded rows:", len(df))

No processed copy found. Creating one...
Loaded rows: 1000


In [ ]:
# Identify Rows That Need Processing
mask = df["Information-Accuracy"].isna() | (df["Information-Accuracy"] == "")
remaining_rows = df[mask]

start_index = len(df) - len(remaining_rows)
print(f"Starting evaluation from row {start_index}/{len(df)}\n")

batch_counter = 0

Starting evaluation from row 0/1000



#### MAIN LOOP

In [ ]:
from tqdm import tqdm
import re
import json

# MAIN EVALUATION LOOP WITH TQDM
for idx in tqdm(remaining_rows.index, desc="Evaluating rows", unit="row"):

    row = df.loc[idx]

    prompt_filled = EVAL_PROMPT.format(
        conversation_history=row["conversation_history"],
        generated_summary=row["generated_summary"],
    )

    try:
        # SEND REQUEST TO CLAUDE
        response = anthropic.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=300,
            temperature=0,
            messages=[{"role": "user", "content": prompt_filled}],
        )

        # EXTRACT RAW TEXT
        raw_text = response.content[0].text.strip()
        if not raw_text:
            raise ValueError("Claude returned an empty response")

        # CLEAN THE JSON TEXT
        cleaned = raw_text.strip()

        cleaned = re.sub(r"^```[a-zA-Z0-9]*\s*", "", cleaned)
        cleaned = re.sub(r"\s*```$", "", cleaned)
        cleaned = re.sub(r"<[^>]+>", "", cleaned).strip()
        cleaned = cleaned.replace("\ufeff", "").replace("\u200b", "").strip("`").strip()

        # TRY PARSING JSON
        try:
            result_json = json.loads(cleaned)
        except json.JSONDecodeError:
            raise ValueError(f"Invalid JSON from Claude:\n{raw_text}")

        # UPDATE DF ONLY IF SUCCESSFUL
        df.at[idx, "Information-Accuracy"] = result_json["Information-Accuracy"]
        df.at[idx, "Structural-Clarity"]   = result_json["Structural-Clarity"]
        df.at[idx, "Faithfulness"]         = result_json["Faithfulness"]

        # SAVE BATCH (SAFE)
        batch_counter += 1

        if batch_counter >= batch_size:
            df.to_csv(processed_csv, index=False, encoding="utf-8-sig")
            print(f"Batch saved (processed up to row {idx}).")
            batch_counter = 0

    except Exception as e:
        # ON ANY ERROR, STOP IMMEDIATELY
        # DO NOT SAVE ANYTHING
        print(f"\nERROR at row {idx}: {e}")
        print("Stopping execution WITHOUT saving this partial batch.")
        break

    time.sleep(batch_pause)

Evaluating rows:   5%|▍         | 49/1000 [02:58<1:04:10,  4.05s/row]

Batch saved (processed up to row 49).


Evaluating rows:  10%|▉         | 99/1000 [06:19<1:02:46,  4.18s/row]

Batch saved (processed up to row 99).


Evaluating rows:  15%|█▍        | 149/1000 [09:26<59:39,  4.21s/row]

...

Evaluating rows:  95%|█████████▍| 949/1000 [56:47<03:09,  3.71s/row]

Batch saved (processed up to row 949).


Evaluating rows: 100%|█████████▉| 999/1000 [59:46<00:03,  3.22s/row]

Batch saved (processed up to row 999).


Evaluating rows: 100%|██████████| 1000/1000 [59:51<00:00,  3.59s/row]


In [ ]:
from datasets import Dataset

repo = "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data"

# Convert to HF Dataset (remove index column if exists)
hf_dataset = Dataset.from_pandas(df.reset_index(drop=True))

# Push to the hub (creates parquet automatically)
hf_dataset.push_to_hub(repo, private=False)


readme_text = """
# customer-service-context-summarization-evaluation-data
Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data
Dataset updated with context summarization evaluation columns.
This README refresh triggers Hugging Face metadata re-index.
"""

with open("README.md", "w") as f:
    f.write(readme_text)

from huggingface_hub import HfApi

api = HfApi()
repo_id = "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data"

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Force metadata refresh – updated README only"
)

print("README uploaded. Metadata will refresh shortly.")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  75%|#######5  | 1.10MB / 1.46MB            

README.md:   0%|          | 0.00/784 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


README uploaded. Metadata will refresh shortly.


In [ ]:
from datasets import load_dataset
import pandas as pd

# Load dataset from Hugging Face
dataset = load_dataset(
    "Lakshan2003/Llama3.1-8b-instruct-customerservice-context-summarization-llm-judge-data",
    split="train"
)

# Convert to pandas DataFrame
df = dataset.to_pandas()

# Correct column names (exact match)
task_columns = [
    "Information-Accuracy",
    "Structural-Clarity",
    "Faithfulness",
]

# Ensure numeric dtype
df[task_columns] = df[task_columns].apply(pd.to_numeric, errors="coerce")

# Compute task-wise mean
task_wise_mean = df[task_columns].mean()

# Convert to clean table
task_wise_mean_df = task_wise_mean.reset_index()
task_wise_mean_df.columns = ["Task", "Mean Score"]

README.md:   0%|          | 0.00/268 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.46M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print(task_wise_mean_df)

                   Task  Mean Score
0  Information-Accuracy    4.689690
1    Structural-Clarity    4.903904
2          Faithfulness    4.629630
